# 🔬 Track 2: RT-DETR-L Training on Kaggle (Background Cloud GPU)
This notebook trains RT-DETR-L on DsPCBSD+ with headless background support.
You can click **Save Version** -> **Save & Run All (Commit)** and close your laptop!

In [ ]:
# 1. Install ultralytics & dependencies
!pip install -q ultralytics ensemble-boxes onnx onnxsim


In [ ]:
# 2. Check GPU & Auto-Discover Dataset
import os, sys, torch
from pathlib import Path

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

# Fast shallow auto-discovery without slow recursive rglob
input_dir = Path("/kaggle/input")
dataset_path = None
candidates = (
    list(input_dir.glob("*/images"))
    + list(input_dir.glob("*/*/images"))
    + list(input_dir.glob("*/*/*/images"))
)
for p in candidates:
    if (p / "train").is_dir() and (p / "val").is_dir():
        dataset_path = p.parent
        break

if dataset_path is None:
    raise FileNotFoundError(f"Could not find images/train under {input_dir}. Did you attach the dataset?")
print(f"Found dataset at: {dataset_path}")


In [ ]:
# 3. Create data.yaml in /kaggle/working
import yaml

DATA_YAML = "/kaggle/working/data.yaml"
data_config = {
    'path': str(dataset_path),
    'train': 'images/train',
    'val': 'images/val',
    'nc': 9,
    'names': {
        0: 'SH', 1: 'SP', 2: 'SC', 3: 'OP', 4: 'MB',
        5: 'HB', 6: 'CS', 7: 'CFO', 8: 'BMFO'
    }
}
with open(DATA_YAML, 'w') as f:
    yaml.safe_dump(data_config, f, sort_keys=False)

print(f"data.yaml generated at: {DATA_YAML}")


In [ ]:
# 4. Initialize & Train RT-DETR-L (Stage 2 Fine-Tuning)
from ultralytics import RTDETR
from pathlib import Path

# Resolve Stage 1 checkpoint
hits = (
    list(Path("/kaggle/input").glob("*/pcb-rtdetr-stage1/best.pt"))
    + list(Path("/kaggle/input").glob("*/*/pcb-rtdetr-stage1/best.pt"))
    + list(Path("/kaggle/input").glob("pcb-rtdetr-stage1/best.pt"))
)
CKPT = str(hits[0]) if hits else "/kaggle/input/datasets/abhinavksinha09/pcb-rtdetr-stage1/best.pt"
print(f"Loading Stage 1 checkpoint: {CKPT}")

model = RTDETR(CKPT)

# Fine-tuning: AdamW, low initial lr, cosine decay, mosaic closure for fine defects
results = model.train(
    data=DATA_YAML,
    epochs=30,
    batch=8,
    imgsz=640,
    device=0,
    optimizer="AdamW",
    lr0=0.00005,
    lrf=0.05,
    cos_lr=True,
    close_mosaic=5,
    project="/kaggle/working/runs/detect",
    name="PCB_RTDETR_L_Stage2",
    exist_ok=True,
    patience=10,
    save=True,
    val=True,
    amp=True
)


In [ ]:
# 5. Detailed Validation & Per-Class Metrics Table
import pandas as pd
from pathlib import Path

best_pt = Path("/kaggle/working/runs/detect/PCB_RTDETR_L_Stage2/weights/best.pt")
eval_model = RTDETR(str(best_pt))
val_res = eval_model.val(data=DATA_YAML, imgsz=640, split="val", device=0)

CLASS_NAMES = ["SH", "SP", "SC", "OP", "MB", "HB", "CS", "CFO", "BMFO"]
metrics_summary = []
for i, name in enumerate(CLASS_NAMES):
    metrics_summary.append({
        "Class": name,
        "Precision": round(float(val_res.box.p[i]), 4),
        "Recall": round(float(val_res.box.r[i]), 4),
        "mAP50": round(float(val_res.box.ap50[i]), 4),
        "mAP50-95": round(float(val_res.box.ap[i]), 4)
    })

df = pd.DataFrame(metrics_summary)
print("=== Per-Class Performance Summary (Stage 2) ===")
print(df.to_string(index=False))
df.to_csv("/kaggle/working/rtdetr_stage2_per_class_metrics.csv", index=False)


In [ ]:
# 6. Export to ONNX for Edge Inference
print("Exporting trained Stage 2 RT-DETR-L model to ONNX format...")
try:
    onnx_path = eval_model.export(format="onnx", imgsz=640, simplify=True)
    print(f"ONNX Model exported: {onnx_path}")
except Exception as e:
    print(f"ONNX export note: {e}")


In [ ]:
# 7. Verification of Output Files in /kaggle/working
import os
print("=== Saved Output Files in /kaggle/working ===")
for root, dirs, files in os.walk("/kaggle/working"):
    for f in files:
        fp = os.path.join(root, f)
        sz_mb = os.path.getsize(fp) / (1024 * 1024)
        if sz_mb > 0.1 or f.endswith(('.pt', '.onnx', '.csv', '.png')):
            rel = os.path.relpath(fp, "/kaggle/working")
            print(f"{rel:50s} ({sz_mb:.2f} MB)")
